# Mini Projet — Partie 1 : Data Process + Data Visualization
## « L'Afrique de l'Ouest est-elle bien desservie par le transport aérien ? »

**UE FDIA — Traitement de données avec Python** | CITADEL / UVBF

---

## Étape 0 — Configuration des paramètres individualisés

In [ ]:
import hashlib

MATRICULE = "N00268120111"  

REGIONS = {
    0: ("Afrique de l'Ouest", ["BF", "ML", "NE", "SN", "CI", "GH", "NG", "TG", "BJ", "GN"]),
}
SCENARIOS = {
    0: ("A", "Comité d'investissement régional (ex. BOAD)",
        "Justifier un investissement dans un pays sous-desservi de votre échantillon"),
}

region_name, region_countries = REGIONS[0]
scenario_code, scenario_who, scenario_what = SCENARIOS[0]

print(f"Région assignée   : {region_name} {region_countries}")
print(f"Scénario assigné  : {scenario_code}")
print(f"  Who (audience)  : {scenario_who}")
print(f"  What (message)  : {scenario_what}")

## 📑 PHASE A — Data Process

### A.1 — Import des données

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv"

df = pd.read_csv(URL)
print(df.shape)
df.head()

### A.2 — Prétraitement : Sélection & Renommer

In [ ]:
# Sélection des colonnes utiles du cahier des charges
colonnes_utiles = ['ident', 'name', 'type', 'iso_country', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'scheduled_service']
df = df[colonnes_utiles]

# Renommage explicite des colonnes selon les consignes
df = df.rename(columns={
    'ident': 'ID',
    'name': 'airport_name',
    'type': 'type',
    'iso_country': 'country',
    'latitude_deg': 'lat',
    'longitude_deg': 'long',
    'elevation_ft': 'elevation',
    'scheduled_service': 'service'
})

df.head()

#### Traitement des valeurs manquantes (`elevation`)

In [ ]:
nb_nan = df["elevation"].isnull().sum()
print(f"Valeurs manquantes dans 'elevation' : {nb_nan} sur {len(df)} lignes")

# Traitement : Remplacement par la valeur médiane propre à chaque pays
df['elevation'] = df.groupby('country')['elevation'].transform(lambda x: x.fillna(x.median()))

# Cas de secours : Si un pays n'a aucune donnée d'altitude, on applique la médiane globale
df['elevation'] = df['elevation'].fillna(df['elevation'].median())

**Justification de la stratégie choisie :** 
L'altitude (`elevation`) dépend directement de la géographie locale d'un pays. Remplacer les données manquantes par une moyenne mondiale fausserait la réalité topographique. Utiliser la **médiane par pays** permet de conserver une cohérence physique locale réaliste. Conserver les lignes contenant ces valeurs manquantes est essentiel pour notre Phase B, car l'analyse repose sur le décompte total des infrastructures par pays, et non sur des modélisations physiques nécessitant une précision absolue de l'altitude.

In [ ]:
df.to_csv("airports_clean.csv", index=False)
print("Fichier 'airports_clean.csv' sauvegardé :", df.shape)

### A.3 — Opérations individualisées (Afrique de l'Ouest)

In [ ]:
# Filtrage sur les pays de l'Afrique de l'Ouest assignés
df_region = df[df['country'].isin(region_countries)].copy()
print(f"Nombre d'aéroports dans la région {region_name} : {len(df_region)}")

# 1. Extraction des aéroports contenant le mot-clé 'International'
df_international = df_region[df_region['airport_name'].str.contains('International', case=False, na=False)]
df_international.to_csv("international.csv", index=False)

# 2. Extraction des aéroports dont le nom commence entre A et M
df_range_alpha = df_region[df_region['airport_name'].str.upper().str.get(0).between('A', 'M')]
df_range_alpha.to_csv("range_alpha.csv", index=False)

# 3. Identification des altitudes extrêmes de la région avec nsmallest / nlargest
alt_min = df_region.nsmallest(1, 'elevation')[['airport_name', 'country', 'elevation']].iloc[0]
alt_max = df_region.nlargest(1, 'elevation')[['airport_name', 'country', 'elevation']].iloc[0]

print(f"\nAltitude minimale : {alt_min['airport_name']} ({alt_min['country']}) avec {alt_min['elevation']} ft")
print(f"Altitude maximale : {alt_max['airport_name']} ({alt_max['country']}) avec {alt_max['elevation']} ft")

#### Table de synthèse par pays

In [ ]:
# Construction de la table de synthèse par pays
synthese = df_region.groupby('country').agg(
    total_airports=('ID', 'count'),
    nb_large=('type', lambda x: (x == 'large_airport').sum()),
    nb_service_regulier=('service', lambda x: (x == 'yes').sum())
).reset_index()

# Identification des pays sans grand aéroport en service régulier
large_actif = df_region[(df_region['type'] == 'large_airport') & (df_region['service'] == 'yes')]
pays_avec_large_actif = set(large_actif['country'].unique())

pays_sous_desservis = [c for c in region_countries if c not in pays_avec_large_actif]
print("Pays sous-desservis (aucun grand aéroport régulier actif) :", pays_sous_desservis)

# Exportation de la synthèse
synthese.to_csv("synthese_pays.csv", index=False)
synthese

## 📊 PHASE B — Data Visualization

### B.1 — Contexte (Who / What / How)
* **Who (audience) :** Le **Comité d'investissement régional de la BOAD**. C'est une audience institutionnelle, technique et financière exigeant des arguments pragmatiques pour arbitrer l'allocation de fonds de développement.
* **What (message) :** Justifier l'octroi prioritaire d'un investissement de modernisation aéroportuaire au **Niger (NE)**, pays de notre échantillon structurellement sous-desservi ne possédant aucun grand aéroport actif connecté aux lignes régulières mondiales.
* **How (mécanisme) :** Un document d'aide à la décision court (*Executive Summary*) adossé à un graphique épuré et mémorisable pour marquer les esprits lors d'un comité d'arbitrage.
* **Ton souhaité :** Factuel, institutionnel, rigoureux et persuasif.

### B.2 — Exploratoire vs Explicatoire
* **Analyse exploratoire :** L'analyse de la table `synthese` montre que le Nigeria (NG) domine largement le volume global d'infrastructures en Afrique de l'Ouest. Cependant, l'indicateur le plus critique pour le développement commercial est l'accès à un service aérien régulier (`nb_service_regulier`). À ce niveau, le **Niger (NE)** affiche un retard criant (seulement 4 aéroports ouverts aux lignes régulières), créant une rupture d'équité territoriale face à des hubs connectés comme la Côte d'Ivoire ou le Ghana.
* **Analyse explicative :** Nous choisissons de transmettre **un seul message** : le Niger est l'angle mort de la connectivité aérienne en Afrique de l'Ouest. C'est l'argument clé pour légitimer l'action de la BOAD dans le cadre de sa mission de désenclavement régional.

### B.3 — Storyboard
1. **Issue :** L'intégration économique de la zone UEMOA est entravée par d'importantes fractures infrastructurelles entre pays côtiers et pays enclavés.
2. **Démonstration de l'issue :** Les données de *OurAirports* révèlent que le Niger ne dispose que de 4 aéroports offrant un service régulier sur l'ensemble de son vaste territoire, le plaçant au bas de l'échelle régionale.
3. **Idée / angle retenu :** Isoler visuellement le Niger (en orange vif) au sein d'un classement ouest-africain (en gris neutre) pour matérialiser instantanément son déficit de connectivité.
4. **Recommandation :** La BOAD doit prioriser le financement de nouvelles infrastructures aéroportuaires au Niger pour sécuriser son désenclavement et stimuler les échanges commerciaux régionaux.

### B.4 — Avant : graphique par défaut

In [ ]:
import matplotlib.pyplot as plt

colonne_a_visualiser = "nb_service_regulier"

fig, ax = plt.subplots()
ax.bar(synthese["country"], synthese[colonne_a_visualiser])
ax.set_title(f"{colonne_a_visualiser} by country")
plt.show()

### B.5 — Choix du graphique (justification)
Le **bar chart horizontal** est le choix recommandé pour comparer des quantités entre catégories (pays). Il offre une lecture naturelle de haut en bas et permet d'afficher les étiquettes de texte (*BF, NE, CI...*) horizontalement sans forcer le lecteur à pencher la tête.

Les *pie charts* et *donut charts* sont proscrits car l'œil humain perçoit très mal les surfaces angulaires et les arcs de cercle, rendant la comparaison complexe entre 10 pays impossible. Les graphiques 3D sont également interdits car la perspective artificielle fausse la hauteur réelle des barres et nuit à la rigueur d'une présentation technique destinée à des décideurs.

### B.6 — Decluttering pas à pas

#### Étape 1 — Passage en barres horizontales + suppression des bordures

In [ ]:
synthese_sorted = synthese.sort_values(colonne_a_visualiser)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser], color="steelblue")

for spine in ax.spines.values():
    spine.set_visible(False)

plt.show()

#### Étape 2 — Nettoyer les labels d'axes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser], color="steelblue")

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0)
ax.set_xticklabels([])
ax.set_xlabel("")
ax.tick_params(axis="y", length=0)

plt.show()

#### Étape 3 — Labellisation directe des données + couleur intentionnelle

In [ ]:
couleurs = ['#D3D3D3' if c != 'NE' else '#FF6F00' for c in synthese_sorted["country"]]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser], color=couleurs)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0)
ax.set_xticklabels([])
ax.tick_params(axis="y", length=0)

for i, bar in enumerate(bars):
    width = bar.get_width()
    country = synthese_sorted["country"].iloc[i]
    is_target = (country == 'NE')
    
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2,
            f'{int(width)}',
            va='center', ha='left', fontsize=10,
            color='#FF6F00' if is_target else '#333333',
            fontweight='bold' if is_target else 'normal')

plt.show()

#### Étape 4 — Version finale : titre déclaratif portant l'insight

In [ ]:
titre_declaratif = "Le Niger, angle mort du transport aérien régulier en Afrique de l'Ouest"

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(synthese_sorted["country"], synthese_sorted[colonne_a_visualiser], color=couleurs)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0)
ax.set_xticklabels([])
ax.tick_params(axis="y", labelsize=11, length=0)

for i, bar in enumerate(bars):
    width = bar.get_width()
    country = synthese_sorted["country"].iloc[i]
    is_target = (country == 'NE')
    
    ax.text(width + 0.6, bar.get_y() + bar.get_height()/2,
            f'{int(width)} aéroports actifs',
            va='center', ha='left', fontsize=10,
            color='#E65100' if is_target else '#555555',
            fontweight='bold' if is_target else 'normal')

ax.set_title(titre_declaratif, fontsize=13, fontweight="bold", loc="left", pad=22, color='#1A237E')

ax.text(0, 1.02, "Nombre d'infrastructures aéroportuaires assurant un service commercial régulier",
        transform=ax.transAxes, fontsize=10, style='italic', color='#555555')

plt.tight_layout()
plt.savefig("graphique_final.png", dpi=150, bbox_inches="tight")
plt.show()

### B.7 — Message final

Au regard de ce diagnostic factuel, le Comité d'investissement de la BOAD est invité à constater le déficit infrastructurel majeur du Niger, qui dispose de seulement 4 plateformes à service régulier pour couvrir l'ensemble de son territoire. Face à cette rupture de connectivité vis-à-vis des hubs de la région, le financement d'un programme de modernisation aéroportuaire au Niger constitue une priorité d'investissement hautement stratégique pour désenclaver la zone sahélienne et dynamiser l'espace économique régional.